# Ders4 - Benchmark Soru Havuzu Analizi

Amaç: Ders2'deki Trendyol scrapper koduyla, **eğitim verisinde kullanılmamış** 6 farklı
ürünün satıcıya-sor bölümünden toplanan alıcı sorusu / ürün bilgisi / satıcı cevabı
üçlülerini analiz ederek, fine-tune edilmiş model ile diğer modelleri karşılaştıracağımız
**tutarlı ve genel bir benchmark soru havuzu** oluşturmak.

Yaklaşım: Ham veriyi hiçbir ön filtreden geçirmeden (şablon tespiti, kural bazlı ayıklama vb.
yapmadan) doğrudan ürün başına tek bir istekte LLM'e (OpenRouter) veriyoruz. Model, aynı
ürüne ait tüm soru-cevap çiftlerini birlikte görerek:
- Şablon/otomatik cevapları (mesai saati, tatil bildirimi, "sorunuz kriterlere uygun değil"
  gibi ürünle ilgisi olmayan cevaplar) eler,
- Anlamca aynı soruları tek bir kanonik soruda birleştirir,
- Aynı bilginin birden çok kez tekrarlanmasından (verideki doğal tekrarlardan) o bilginin
  tutarlı olduğunu kendisi çıkarır,

ve doğrudan temiz bir JSON soru havuzu döndürür. Ayrıştırma/kümeleme mantığını elle
kurmak yerine bu kararı uçtan uca LLM'e bırakıyoruz.


In [9]:
## Gerekli paketleri kuruyoruz (ilk çalıştırmada bir kere yeterli).
!uv pip install -q pandas httpx matplotlib


In [10]:
import asyncio
import json
import os
import re

import pandas as pd


def ders4_dizinini_bul():
    """Jupyter kernel'inin çalışma dizini nereden başlatıldığından (repo kökü, Ders4'ün
    kendisi vb.) bağımsız olarak Ders4 klasörünü bulur, böylece göreli yollar her zaman
    doğru çalışır."""
    isaret_dosya = "BenchmarkSoruHavuzuAnalizi.ipynb"
    baslangic = os.getcwd()

    adaylar = [baslangic, os.path.join(baslangic, "Ders4")]
    ust = baslangic
    for _ in range(5):
        ust = os.path.dirname(ust)
        adaylar.append(os.path.join(ust, "Ders4"))

    for aday in adaylar:
        if os.path.exists(os.path.join(aday, isaret_dosya)):
            return aday

    raise FileNotFoundError(
        f"'{isaret_dosya}' bulunamadı. Notebook'u 'Ders4' klasöründen ya da onu içeren "
        f"proje kökünden çalıştırdığından emin ol (şu an çalışma dizini: {baslangic})."
    )


os.chdir(ders4_dizinini_bul())
print("Çalışma dizini:", os.getcwd())

# Ham veri Ders2 ödevinde toplandığı için oradaki .env dosyasını (OPENROUTER_* anahtarları)
# tekrar tanımlamak yerine doğrudan kullanıyoruz.
VERI_KLASORU = "../Ders2/DataCollection-Scrapping/json_ciktilari"
ENV_DOSYASI = "../Ders2/DataCollection-Scrapping/.env"

LLM_SONUC_DOSYASI = "llm_urun_bazli_sonuclar.json"
BENCHMARK_HAVUZU_DOSYASI = "benchmark_soru_havuzu.json"


def env_dosyasini_yukle(yol):
    if not os.path.exists(yol):
        return
    with open(yol, "r", encoding="utf-8") as f:
        for satir in f:
            satir = satir.strip()
            if not satir or satir.startswith("#") or "=" not in satir:
                continue
            anahtar, _, deger = satir.partition("=")
            os.environ.setdefault(anahtar.strip(), deger.strip().strip('"').strip("'"))


env_dosyasini_yukle(ENV_DOSYASI)

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.environ.get("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "anthropic/claude-3.5-sonnet")

print("Model:", OPENROUTER_MODEL, "| API key tanımlı mı:", bool(OPENROUTER_API_KEY))


Çalışma dizini: /Users/salih/Desktop/magibu-work/Ders4
Model: google/gemini-2.5-flash-lite:nitro | API key tanımlı mı: True


## 1. Veriyi Yükleme

In [11]:
def diyaloglari_yukle(klasor):
    kayitlar = []
    for dosya_adi in sorted(os.listdir(klasor)):
        if not dosya_adi.endswith(".json"):
            continue
        urun_id = dosya_adi.replace(".json", "")
        with open(os.path.join(klasor, dosya_adi), "r", encoding="utf-8") as f:
            diyaloglar = json.load(f)
        for diyalog in diyaloglar:
            system_msg, user_msg, assistant_msg = diyalog[0], diyalog[1], diyalog[2]
            urun_aciklamasi = system_msg["content"].split("Ürün Özellikleri:", 1)[-1].strip()
            kayitlar.append({
                "urun_id": urun_id,
                "urun_aciklamasi": urun_aciklamasi,
                "soru": user_msg["content"].strip(),
                "cevap": assistant_msg["content"].strip(),
            })
    return pd.DataFrame(kayitlar)


df = diyaloglari_yukle(VERI_KLASORU)
print(f"Toplam {len(df)} soru-cevap çifti, {df['urun_id'].nunique()} ürün.")
df.groupby("urun_id").size().rename("adet")


Toplam 1201 soru-cevap çifti, 6 ürün.


urun_id
1071849383    215
1132636729     31
130582666     248
205133913     248
4128875       248
97879098      211
Name: adet, dtype: int64

## 2. LLM'e Ürün Bazlı Soru Havuzu Oluşturma

Her ürün için: ürün açıklaması + o ürüne ait **tüm ham soru-cevap çiftleri** (hiçbir ön
filtre uygulanmadan, tekrarlar dahil) tek bir promptta LLM'e veriliyor. Model bunları
inceleyip doğrudan temiz bir JSON soru havuzu döndürüyor. Sonuçlar `llm_urun_bazli_sonuclar.json`
dosyasına ürün bazında kaydedilir, böylece işlem yarıda kesilirse kaldığı yerden devam edilebilir.

In [12]:
KATEGORILER = [
    "malzeme", "olcu", "renk", "aksesuar_parca", "stok_varyant",
    "kargo_lojistik", "garanti", "kurulum", "diger",
]

LLM_SISTEM_PROMPTU = f"""Sen bir e-ticaret ürün soru-cevap verisinden benchmark soru havuzu \
hazırlayan bir analistsin. Sana bir ürünün özellikleri ve o ürüne alıcılar tarafından \
sorulmuş TÜM soru-cevap çiftleri (satıcının verdiği ham, filtrelenmemiş cevaplarla birlikte) \
verilecek.

Görevin:
1. Mesai saati, tatil bildirimi, "sorunuz kriterlere uygun değil" gibi ürünle ilgisi olmayan \
   şablon/otomatik cevapları tamamen ele.
2. Anlamca aynı olan soruları tek bir kanonik soruda birleştir (aynı sorunun farklı \
   ifadelerini tekrar tekrar havuza koyma).
3. Bir bilginin veri içinde birden çok kez tutarlı şekilde tekrarlanması, o bilginin \
   güvenilir olduğuna işarettir; çelişkili cevaplar varsa en sık ve en açıklayıcı olanı seç.
4. Sonucu ürün başına en fazla 25 soruyla sınırla; en bilgilendirici ve tekrar eden \
   sorulara öncelik ver.

Kategori seçenekleri: {", ".join(KATEGORILER)}

SADECE aşağıdaki formatta geçerli bir JSON dizisi döndür, başka hiçbir açıklama/önsöz/kod \
bloğu ekleme:
[{{"kategori": "...", "soru": "...", "referans_cevap": "..."}}, ...]"""


def llm_kullanici_promptu(urun_aciklamasi, qa_ciftleri):
    satirlar = [f"Ürün Özellikleri:\n{urun_aciklamasi}\n", "Soru-Cevap Çiftleri:"]
    for i, (soru, cevap) in enumerate(qa_ciftleri, start=1):
        satirlar.append(f"{i}. Soru: {soru}\n   Cevap: {cevap}")
    return "\n".join(satirlar)


def json_govdesini_ayikla(metin):
    """Modelin bazen kod bloğu içine sardığı JSON çıktısını güvenli şekilde çıkarır."""
    metin = metin.strip()
    if metin.startswith("```"):
        metin = re.sub(r"^```[a-zA-Z]*\n?", "", metin)
        metin = re.sub(r"```$", "", metin).strip()
    eslesme = re.search(r"\[.*\]", metin, re.DOTALL)
    return eslesme.group(0) if eslesme else metin


In [13]:
import httpx


async def openrouter_soru_havuzu_uret(client, urun_aciklamasi, qa_ciftleri, max_deneme=3):
    payload = {
        "model": OPENROUTER_MODEL,
        "messages": [
            {"role": "system", "content": LLM_SISTEM_PROMPTU},
            {"role": "user", "content": llm_kullanici_promptu(urun_aciklamasi, qa_ciftleri)},
        ],
        "temperature": 0.0,
        "max_tokens": 4000,
    }
    url = OPENROUTER_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    son_hata = None
    for deneme in range(1, max_deneme + 1):
        try:
            resp = await client.post(url, json=payload, headers=headers, timeout=120)
            resp.raise_for_status()
            icerik = resp.json()["choices"][0]["message"]["content"]
            return json.loads(json_govdesini_ayikla(icerik))
        except Exception as e:
            son_hata = str(e)
            await asyncio.sleep(2 * deneme)
    print(f"  Uyarı: analiz başarısız oldu, atlanıyor. Hata: {son_hata}")
    return None


In [14]:
def onceki_sonuclari_yukle():
    if os.path.exists(LLM_SONUC_DOSYASI):
        with open(LLM_SONUC_DOSYASI, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def sonuclari_kaydet(sonuclar):
    tmp = LLM_SONUC_DOSYASI + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(sonuclar, f, ensure_ascii=False, indent=2)
    os.replace(tmp, LLM_SONUC_DOSYASI)


async def tum_urunleri_isle(df, max_eszamanli=6):
    sonuclar = onceki_sonuclari_yukle()
    sem = asyncio.Semaphore(max_eszamanli)

    async with httpx.AsyncClient() as client:
        async def bir_urunu_isle(urun_id, grup):
            if urun_id in sonuclar and sonuclar[urun_id]:
                print(f"  {urun_id}: zaten işlenmiş, atlanıyor.")
                return
            urun_aciklamasi = grup["urun_aciklamasi"].iloc[0]
            qa_ciftleri = list(zip(grup["soru"], grup["cevap"]))
            async with sem:
                havuz = await openrouter_soru_havuzu_uret(client, urun_aciklamasi, qa_ciftleri)
            sonuclar[urun_id] = havuz or []
            sonuclari_kaydet(sonuclar)
            print(f"  {urun_id}: {len(sonuclar[urun_id])} soru üretildi ({len(qa_ciftleri)} ham çiftten).")

        await asyncio.gather(*(
            bir_urunu_isle(urun_id, grup) for urun_id, grup in df.groupby("urun_id")
        ))

    return sonuclar


assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY bulunamadı, Ders2/.env dosyasını kontrol et."
llm_sonuclari = await tum_urunleri_isle(df)
print("LLM analizi tamamlandı.")


  1071849383: zaten işlenmiş, atlanıyor.
  1132636729: zaten işlenmiş, atlanıyor.
  130582666: zaten işlenmiş, atlanıyor.
  205133913: zaten işlenmiş, atlanıyor.
  4128875: zaten işlenmiş, atlanıyor.
  97879098: zaten işlenmiş, atlanıyor.
LLM analizi tamamlandı.


## 3. Sonuçların Birleştirilmesi ve Kaydedilmesi

Ürün bazlı LLM çıktıları tek bir listede birleştirilip benchmark değerlendirme adımında
kullanılacak `benchmark_soru_havuzu.json` dosyasına yazılır.

In [15]:
benchmark_havuzu = []
for urun_id, sorular in llm_sonuclari.items():
    for soru_kaydi in (sorular or []):
        if not isinstance(soru_kaydi, dict):
            continue
        benchmark_havuzu.append({
            "urun_id": urun_id,
            "kategori": soru_kaydi.get("kategori", "diger"),
            "soru": soru_kaydi.get("soru", "").strip(),
            "referans_cevap": soru_kaydi.get("referans_cevap", "").strip(),
        })

with open(BENCHMARK_HAVUZU_DOSYASI, "w", encoding="utf-8") as f:
    json.dump(benchmark_havuzu, f, ensure_ascii=False, indent=2)

print(f"Benchmark soru havuzu kaydedildi: {BENCHMARK_HAVUZU_DOSYASI} ({len(benchmark_havuzu)} soru)")


Benchmark soru havuzu kaydedildi: benchmark_soru_havuzu.json (156 soru)
